In [1]:
import os

target_dir = "../data/raw/financial-fraud-detection-dataset"
csv_file = os.path.join(target_dir, "Synthetic_Financial_datasets_log.csv")

In [2]:
import pandas as pd

df = pd.read_csv(csv_file)
df.shape

(6362620, 11)

In [3]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

Drop unusable columns due to Kaggle rule

In [4]:
df.drop(['oldbalanceOrg', 'newbalanceOrig','newbalanceDest','oldbalanceDest'], axis=1, inplace=True)
df.shape

(6362620, 7)

Confirming that merchants as nameDest do not exhibit fraud

In [5]:
merchants_only = df[df['nameDest'].str.contains("M")]

print(f'Unique values for \'isFraud\' column: {merchants_only['isFraud'].unique()}')
print(f'Unique values for \'isFlaggedFraud\' column: {merchants_only['isFlaggedFraud'].unique()}')

Unique values for 'isFraud' column: [0]
Unique values for 'isFlaggedFraud' column: [0]


No merchants as source accounts

In [6]:
merchantsOrig = df[df['nameOrig'].str.contains("M")]

merchantsOrig.shape

(0, 7)

Remove transactions where nameDest are Merchants

In [7]:
filtered_df = df[~df['nameDest'].str.contains("M")]

print(filtered_df.shape)
df = filtered_df    #set back dataset to 'df' variable name for ease of usage

(4211125, 7)


Place dataset in directory for other models to use

In [8]:
# Create the data/raw directory if it doesn't exist
os.makedirs("../data/raw(without Merchants)", exist_ok=True)

target_dir = "../data/raw(without Merchants)"
full_file_path = os.path.join(target_dir, 'Dataset without Merchants.csv')

df.to_csv(full_file_path, index=False)

In [9]:
# sample = df.sample(frac=0.1, random_state=42)

Identify accounts appearing both as nameOrig and nameDest

In [10]:
senders = set(df['nameOrig'].unique())
receivers = set(df['nameDest'].unique())


overlap_accounts = senders.intersection(receivers)

print(f"Total unique senders: {len(senders)}")
print(f"Total unique receivers: {len(receivers)}")
print(f"Accounts appearing as both sender and receiver: {len(overlap_accounts)}")
print(f'\n')


suspicious_df = df[df['nameOrig'].isin(overlap_accounts) | df['nameDest'].isin(overlap_accounts)]

Total unique senders: 4207035
Total unique receivers: 571961
Accounts appearing as both sender and receiver: 1152




Rate of transactions involve overlapping accounts

In [11]:
total_txn = df.shape[0]
overlap_txn = suspicious_df.shape[0]
print(f"Percentage of transactions involving dual-role accounts: {(overlap_txn / total_txn) * 100:.2f}%")

# Fraud rate among overlapping accounts
fraud_overlap = suspicious_df['isFraud'].sum()
fraud_rate_overlap = fraud_overlap / suspicious_df.shape[0]
overall_fraud_rate = df['isFraud'].mean()

print(f"Fraud rate (overlap accounts): {fraud_rate_overlap:.4f}")
print(f"Overall fraud rate (entire dataset): {overall_fraud_rate:.4f}")

Percentage of transactions involving dual-role accounts: 0.23%
Fraud rate (overlap accounts): 0.0013
Overall fraud rate (entire dataset): 0.0020


In [12]:
suspicious_df['type'].value_counts()

type
CASH_OUT    5219
CASH_IN     3225
TRANSFER    1218
DEBIT         91
Name: count, dtype: int64

In [22]:
# Get the index (nameDest) of top irregular receivers
i = suspicious_df

# Print each irregular receiver
for customer in i['nameOrig']:    
    # Get transactions for this receiver
    # receiver_txns = df[df['nameOrig'] == customer | df['nameDest'] == customer]
    as_sender = df[df['nameOrig'] == customer]
    as_receiver = df[df['nameDest'] == customer]


    all_txns = pd.concat([as_sender,as_receiver])
    if all_txns['isFraud'].sum() == 0:
        continue
    
    print(f"Receiver ID: {customer}")
    print(f"Number of transactions: {len(all_txns)}")
    print(f"Fraud transactions: {all_txns['isFraud'].sum()}")
    print(f"Fraud rate: {(all_txns['isFraud'].sum() / len(all_txns)) * 100:.2f}%")
    print("-" * 50)

Receiver ID: C1496808239
Number of transactions: 1
Fraud transactions: 1
Fraud rate: 100.00%
--------------------------------------------------
Receiver ID: C2086292862
Number of transactions: 2
Fraud transactions: 1
Fraud rate: 50.00%
--------------------------------------------------
Receiver ID: C1829721095
Number of transactions: 1
Fraud transactions: 1
Fraud rate: 100.00%
--------------------------------------------------
Receiver ID: C1175896731
Number of transactions: 1
Fraud transactions: 1
Fraud rate: 100.00%
--------------------------------------------------
Receiver ID: C145966586
Number of transactions: 1
Fraud transactions: 1
Fraud rate: 100.00%
--------------------------------------------------
Receiver ID: C1023330867
Number of transactions: 2
Fraud transactions: 1
Fraud rate: 50.00%
--------------------------------------------------
Receiver ID: C1791019967
Number of transactions: 1
Fraud transactions: 1
Fraud rate: 100.00%
----------------------------------------------

KeyboardInterrupt: 

In [23]:
# Count number of transactions each user has sent
num_sent = df.groupby('nameOrig').size().reset_index(name='numSent')

# Count number of transactions each user has received
num_received = df.groupby('nameDest').size().reset_index(name='numReceived')

# Merge 'numSent' into main df
df = df.merge(num_sent, on='nameOrig', how='left')

# Merge 'numReceived' into main df
df = df.merge(num_received, on='nameDest', how='left')

# Fill NaN (e.g., users who never received anything) with 0
df[['numSent', 'numReceived']] = df[['numSent', 'numReceived']].fillna(0)
